# 08 · Train a fixed, validation-selected configuration

An alternative to the search in notebook 01. The decision-tree hyper-parameters are read from `configs/nf_ton_iot_v2_fixed_params.json`
(gini, `max_depth` 20, `min_samples_split` 10, `min_samples_leaf` 8, no class weights, `ccp_alpha` 1e-6). They were selected by validation MCC in an earlier search
on NF-ToN-IoT-v2 and are re-evaluated here under the leakage-safe feature set. Same protocol as `main()` in notebook 01, same split, same seed:

1. Loads the data with the leakage-safe feature policy and re-creates the identical train / validation / test split.
2. Trains the configuration for several feature counts **K** on the training split and scores them on the **validation** split.
3. Selects **K by validation MCC** only.
4. Refits on train + validation, evaluates the **test split once**, and saves the artifact in the same format as notebook 01
   (`esp32_dt_final_results/<dataset>/full_pipeline_*.pkl`, `metadata_*.json`, `validation_summary_*.csv`), so notebooks 02, 03, 05 and 06 work unchanged.
5. If a previously saved model exists, compares the two **on the validation split** and stops unless validation favours the new one (`FIXED_FORCE=1` overrides).
6. Repeats the selected configuration on independent splits (mean ± std).

Saved models are never deleted; the newest timestamp is picked up by the later notebooks.

In [ ]:
# ---- Setup: find the package root, work inside results/, import the shared code ----
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "framework.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the paper_package folder.")
    ROOT = ROOT.parent
os.environ["PAPER_PACKAGE_ROOT"] = str(ROOT)

# >>> WHICH DATASET?  nf_ton_iot_v2 | cic_ton_iot | nf_unsw_nb15_v3
# run_all.py sets ACTIVE_DATASET; when running by hand, edit the default below
os.environ.setdefault("ACTIVE_DATASET", "nf_unsw_nb15_v3")

sys.path.insert(0, str(ROOT / "src"))
(ROOT / "results").mkdir(exist_ok=True)
os.chdir(ROOT / "results")            # every output (csv / png / pkl / headers) lands under results/

from run_settings import *            # PAPER_MODE: full | quick | smoke

In [ ]:
from framework import *                      # config, pipeline builders, metrics, data helpers
from framework import _json_default
import pandas as pd, numpy as np, matplotlib.pyplot as plt, gc
from datetime import datetime
print("Dataset:", ACTIVE_DATASET, "| CSV:", DEFAULT_DATASET_PATH)
assert Path(DEFAULT_DATASET_PATH).exists(), f"CSV not found: {DEFAULT_DATASET_PATH}"

## 1 · Configuration  *(edit here or use environment variables)*

In [ ]:
FIXED_PARAMS_FILE = os.environ.get("FIXED_PARAMS_FILE", "configs/nf_ton_iot_v2_fixed_params.json")   # relative to the repository root
_pf = Path(FIXED_PARAMS_FILE); _pf = _pf if _pf.is_absolute() else ROOT / _pf
src_meta = json.load(open(_pf, encoding="utf-8"))
FIXED_PARAMS = dict(src_meta["best_params"])
K_CANDIDATES = sorted({int(k) for k in os.environ.get("FIXED_K_LIST", "20,25,30,32,34,36").split(",")})
print("hyper-parameters taken from:", _pf.name)
print("  ", FIXED_PARAMS)
print("  (the source search used K =", src_meta["best_k"], "of", src_meta.get("n_numeric_features", len(src_meta.get("numeric_cols", []))), "features)")
print("K candidates:", K_CANDIDATES, "| rows used to compare K:", f"{FIXED_K_ROWS:,}" if FIXED_K_ROWS else "all training rows")
if ACTIVE_DATASET != "nf_ton_iot_v2":
    print("WARNING: these hyper-parameters were found on NF-ToN-IoT-v2; for another dataset use FIXED_PARAMS_FILE with that dataset's own metadata.")

## 2 · Data and the identical split

In [ ]:
X, y, le, numeric_cols, original_shape = load_and_prepare_dataframe(DEFAULT_DATASET_PATH)
class_names = list(le.classes_)
print(f"shape {original_shape} | {len(numeric_cols)} numeric features | {len(class_names)} classes")

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=VAL_SIZE, stratify=y_train_val, random_state=SEED)
del X_train_val, y_train_val; gc.collect()
rare_classes, extremely_rare_classes = infer_rare_classes(y_train)
print(f"train {len(X_train):,} | validation {len(X_val):,} | test {len(X_test):,}  (test untouched until step 5)")
print("rare classes:", sorted(rare_classes), "| extremely rare:", sorted(extremely_rare_classes))

# consistency with the model currently saved by notebook 01 (same features?)
d = Path("esp32_dt_final_results") / ACTIVE_DATASET
prev_meta_files = sorted(d.glob("metadata_*.json"))
PREV = json.loads(prev_meta_files[-1].read_text(encoding="utf-8")) if prev_meta_files else None
if PREV:
    same = sorted(PREV["numeric_cols"]) == sorted(numeric_cols)
    print(f"previous model: {prev_meta_files[-1].name} | same feature set as the previous model:", "yes" if same else "WARNING: NO")

## 3 · Compare K on the **validation** split  *(no test data)*

In [ ]:
Xk, yk = stratified_sample(X_train, y_train, FIXED_K_ROWS, SEED) if FIXED_K_ROWS else (X_train, y_train)
print(f"fitting each K on {len(Xk):,} training rows; scoring on {len(X_val):,} validation rows")
cand = []
for k in K_CANDIDATES:
    k_eff = min(k, len(numeric_cols)); seed_k = SEED + k_eff
    if any(c["k"] == k_eff for c in cand): continue
    t0 = datetime.now()
    pipe = build_pipeline(FIXED_PARAMS, k_eff, seed_k); pipe.fit(Xk, yk)
    vm = evaluate_on_split(pipe, X_val, y_val, rare_classes, extremely_rare_classes)
    cand.append({"k": k_eff, "validation_mcc": vm["mcc"], "validation_accuracy": vm["accuracy"], "validation_macro_f1": vm["macro_f1"],
                 "validation_small_cls_f1": vm["small_class_f1"], "model_size_kb_proxy": vm["embedded_size_kb"], "nodes": vm["node_count"], "metrics": vm})
    print(f"  K={k_eff:>2}  val MCC {vm['mcc']:.4f}  macro-F1 {vm['macro_f1']:.4f}  nodes {vm['node_count']:>7,}  ({(datetime.now()-t0).seconds}s)")
    del pipe; gc.collect()

cand_df = pd.DataFrame([{k: v for k, v in c.items() if k != "metrics"} for c in cand]); display(cand_df.round(4))
best = max(cand, key=lambda c: (round(c["validation_mcc"], 6), -c["nodes"]))     # ties -> smaller tree
BEST_K = best["k"]
print(f"\n🏆 selected K = {BEST_K}  (validation MCC {best['validation_mcc']:.4f})")

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(cand_df["k"], cand_df["validation_mcc"], "o-", label="validation MCC"); ax.plot(cand_df["k"], cand_df["validation_macro_f1"], "s--", label="validation macro-F1")
ax.axvline(BEST_K, color="gray", ls=":"); ax.set_xlabel("selected features K"); ax.set_ylabel("score"); ax.legend(loc="lower right"); ax.set_title(f"{ACTIVE_DATASET}: fixed configuration, validation score vs K")
fig.savefig(FIGURES_DIR / "validation_k_sweep_fixed_config.png", dpi=300, bbox_inches="tight"); plt.show()

## 4 · Validation metrics of the selected K on the **full** training split
(only needed when K was compared on a subset above; these are the validation numbers stored in the metadata, comparable with the previous model's)

In [ ]:
if FIXED_K_ROWS and len(X_train) > FIXED_K_ROWS:
    vp = build_pipeline(FIXED_PARAMS, BEST_K, SEED + BEST_K); vp.fit(X_train, y_train)
    best_val_metrics = evaluate_on_split(vp, X_val, y_val, rare_classes, extremely_rare_classes); del vp; gc.collect()
else:
    best_val_metrics = best["metrics"]
print(f"selected model, validation (fitted on the full training split): MCC {best_val_metrics['mcc']:.4f} | macro-F1 {best_val_metrics['macro_f1']:.4f} | accuracy {best_val_metrics['accuracy']:.4f} | nodes {best_val_metrics['node_count']:,}")
if PREV:
    pv = PREV["validation_metrics"]
    print(f"previous model,                      validation: MCC {pv['mcc']:.4f} | macro-F1 {pv['macro_f1']:.4f} | accuracy {pv['accuracy']:.4f}")
    if best_val_metrics["mcc"] <= pv["mcc"] and os.environ.get("FIXED_FORCE") != "1":
        raise RuntimeError(f"STOP: validation does NOT favour the new configuration (validation MCC {best_val_metrics['mcc']:.4f} <= previous {pv['mcc']:.4f}). "
                           "Nothing was saved. Set FIXED_FORCE=1 to save anyway.")
    print(f"\nvalidation favours the new configuration (+{best_val_metrics['mcc'] - pv['mcc']:.4f} MCC) -> saving the new model.")

## 5 · Final model: refit on train + validation, evaluate the test set **once**, save

In [ ]:
X_final = pd.concat([X_train, X_val], axis=0); y_final = np.concatenate([y_train, y_val])
final_pipe = build_pipeline(FIXED_PARAMS, BEST_K, SEED); final_pipe.fit(X_final, y_final)
test_metrics = evaluate_on_split(final_pipe, X_test, y_test, rare_classes, extremely_rare_classes)
print("🎯 final test results (reported once):")
for k_, lab in [("accuracy", "Accuracy"), ("macro_f1", "Macro F1"), ("mcc", "MCC"), ("kappa", "Kappa"), ("small_class_f1", "Small-class F1"),
                ("minority_recall", "Minority recall"), ("rare_class_mcc", "Rare-class MCC"), ("node_count", "Nodes"), ("depth", "Depth")]:
    print(f"  {lab:16s}: {test_metrics[k_]:.4f}" if isinstance(test_metrics[k_], float) else f"  {lab:16s}: {test_metrics[k_]}")

output_dir = Path("esp32_dt_final_results") / ACTIVE_DATASET; output_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
artifact = {"pipeline": final_pipe, "label_encoder": le, "numeric_cols": numeric_cols, "target_col": TARGET_COL,
            "dropped_cols": DROP_COLS_IF_PRESENT + IP_COLS_IF_PRESENT, "rare_classes": sorted(rare_classes), "extremely_rare_classes": sorted(extremely_rare_classes)}
model_path = output_dir / f"full_pipeline_{stamp}.pkl"
with open(model_path, "wb") as fh: pickle.dump(artifact, fh)
try: final_feats = list(final_pipe.named_steps["selector"].get_feature_names_out())
except Exception: final_feats = []
scalar = lambda dct: {k: v for k, v in dct.items() if isinstance(v, (int, float, bool, str, type(None)))}
metadata = {
    "selection_rule": "fixed hyper-parameters (from an earlier search run); K chosen by validation MCC; test evaluated once after selection",
    "config_source": str(_pf.name), "config_source_note": "earlier search run on the 41-feature set, selected by validation MCC; re-evaluated here on the leakage-safe feature policy",
    "best_k": BEST_K, "best_params": FIXED_PARAMS,
    "validation_metrics": scalar(best_val_metrics), "test_metrics": scalar(test_metrics),
    "k_candidates_validation": [{k: v for k, v in c.items() if k != "metrics"} for c in cand],
    "numeric_cols": numeric_cols, "selected_features_final": final_feats, "classes": le.classes_.tolist(),
    "rare_classes": sorted(rare_classes), "extremely_rare_classes": sorted(extremely_rare_classes), "model_path": str(model_path),
    "previous_model_metadata": prev_meta_files[-1].name if PREV else None,
    "note": "ESP32 size/latency values are host-side proxy estimates. Always benchmark the generated embedded C on real hardware before deployment."}
meta_path = output_dir / f"metadata_{stamp}.json"
json.dump(metadata, open(meta_path, "w", encoding="utf-8"), indent=2, default=_json_default)
cand_df[["k", "validation_mcc", "validation_accuracy", "validation_macro_f1", "validation_small_cls_f1", "model_size_kb_proxy"]].to_csv(output_dir / f"validation_summary_{stamp}.csv", index=False)
print("\nartifact :", model_path.name, "\nmetadata :", meta_path.name)

In [ ]:
# ---- side-by-side with the previous model + a provenance report for the paper ----------
rep = {"new": {"file": meta_path.name, "k": BEST_K, "val_mcc": best_val_metrics["mcc"], "val_macro_f1": best_val_metrics["macro_f1"],
               "test_mcc": test_metrics["mcc"], "test_macro_f1": test_metrics["macro_f1"], "test_accuracy": test_metrics["accuracy"], "nodes": test_metrics["node_count"]}}
rows = [("new (fixed config)", BEST_K, best_val_metrics["mcc"], best_val_metrics["macro_f1"], test_metrics["mcc"], test_metrics["macro_f1"], test_metrics["accuracy"], test_metrics["node_count"])]
if PREV:
    pt, pv = PREV["test_metrics"], PREV["validation_metrics"]
    rows.append(("previous (searched)", PREV["best_k"], pv["mcc"], pv["macro_f1"], pt["mcc"], pt["macro_f1"], pt["accuracy"], pt["node_count"]))
    rep["previous"] = {"file": prev_meta_files[-1].name, "k": PREV["best_k"], "val_mcc": pv["mcc"], "val_macro_f1": pv["macro_f1"], "test_mcc": pt["mcc"],
                       "test_macro_f1": pt["macro_f1"], "test_accuracy": pt["accuracy"], "nodes": pt["node_count"], "params": PREV["best_params"]}
cmp = pd.DataFrame(rows, columns=["model", "K", "val MCC", "val macro-F1", "test MCC", "test macro-F1", "test accuracy", "nodes"]); display(cmp.round(4))
rep["selection"] = "by validation MCC only; test evaluated once"; rep["hyperparameters"] = FIXED_PARAMS; rep["hyperparameter_source"] = _pf.name
json.dump(rep, open(output_dir / f"fixed_config_report_{stamp}.json", "w", encoding="utf-8"), indent=2, default=_json_default)

## 6 · Multi-seed robustness  *(same protocol as notebook 01: the selected configuration is re-trained and re-tested on independent splits)*

In [ ]:
# =============================================================================
# MULTI-SEED ROBUSTNESS CHECK
# =============================================================================
N_REPEAT_SEEDS = N_REPEAT_SEEDS_CFG
REPEAT_SEEDS = [SEED + 100 + i for i in range(N_REPEAT_SEEDS)]  # distinct from SEED itself

print(f"Dataset profile : {ACTIVE_DATASET!r}")
print(f"Reusing search result: best_k={metadata['best_k']}, best_params={metadata['best_params']}")
print(f"Running {N_REPEAT_SEEDS} independent seeds: {REPEAT_SEEDS}\n")

# Reload once; only the split random_state changes per seed below.
X_ms, y_ms, le_ms, numeric_cols_ms, _shape_ms = load_and_prepare_dataframe(DEFAULT_DATASET_PATH)

repeat_rows = []
for seed_i in REPEAT_SEEDS:
    X_train_val_i, X_test_i, y_train_val_i, y_test_i = train_test_split(
        X_ms, y_ms, test_size=TEST_SIZE, stratify=y_ms, random_state=seed_i
    )
    X_train_i, X_val_i, y_train_i, y_val_i = train_test_split(
        X_train_val_i, y_train_val_i,
        test_size=VAL_SIZE, stratify=y_train_val_i, random_state=seed_i
    )
    X_final_i = pd.concat([X_train_i, X_val_i], axis=0)
    y_final_i = np.concatenate([y_train_i, y_val_i])

    rare_i, extremely_rare_i = infer_rare_classes(y_train_i)

    pipe_i = build_pipeline(metadata["best_params"], metadata["best_k"], seed_i)
    pipe_i.fit(X_final_i, y_final_i)
    metrics_i = evaluate_on_split(pipe_i, X_test_i, y_test_i, rare_i, extremely_rare_i)

    row = {"seed": seed_i}
    row.update({k: v for k, v in metrics_i.items() if isinstance(v, (int, float, bool))})
    repeat_rows.append(row)

    print(f"  seed {seed_i}: accuracy={metrics_i['accuracy']:.4f}  "
          f"macro_f1={metrics_i['macro_f1']:.4f}  mcc={metrics_i['mcc']:.4f}  "
          f"minority_recall={metrics_i.get('minority_recall', float('nan')):.4f}")
    gc.collect()

repeats_df = pd.DataFrame(repeat_rows)

METRIC_COLS_FOR_SUMMARY = [c for c in [
    "accuracy", "macro_f1", "weighted_f1", "mcc", "kappa",
    "small_class_f1", "minority_recall", "rare_class_mcc",
    "extremely_rare_mcc", "embedded_size_kb", "latency_ms_per_sample",
] if c in repeats_df.columns]

summary_stats = repeats_df[METRIC_COLS_FOR_SUMMARY].agg(["mean", "std"]).T
summary_stats.columns = ["mean", "std"]

print(f"\nMulti-seed test-set summary (mean ± std over {N_REPEAT_SEEDS} seeds):")
for m in METRIC_COLS_FOR_SUMMARY:
    print(f"  {m:22s}: {summary_stats.loc[m, 'mean']:.4f} ± {summary_stats.loc[m, 'std']:.4f}")

MULTISEED_DIR = Path("article_results") / ACTIVE_DATASET
MULTISEED_DIR.mkdir(parents=True, exist_ok=True)
repeats_df.to_csv(MULTISEED_DIR / "multi_seed_repeats_raw.csv", index=False)
summary_stats.to_csv(MULTISEED_DIR / "multi_seed_repeats_summary.csv")

# Fold mean/std into article_results_summary.json if that file already
# exists for this run (safe no-op otherwise -- run the Article evidence
# package cell before or after this one, order doesn't matter).
summary_json_path = MULTISEED_DIR / "article_results_summary.json"
if False:   # disabled: notebook 02 regenerates article_results_summary.json for the new model
    with open(summary_json_path, "r", encoding="utf-8") as f:
        existing_summary = json.load(f)
    existing_summary["multi_seed_n"] = N_REPEAT_SEEDS
    existing_summary["multi_seed_test_metrics_mean_std"] = {
        m: {"mean": float(summary_stats.loc[m, "mean"]), "std": float(summary_stats.loc[m, "std"])}
        for m in METRIC_COLS_FOR_SUMMARY
    }
    with open(summary_json_path, "w", encoding="utf-8") as f:
        json.dump(existing_summary, f, indent=2, default=str)
    print(f"\nUpdated {summary_json_path} with multi-seed mean/std.")
else:
    print(f"\nNote: {summary_json_path} does not exist yet. Run the Article evidence "
          "package cell (below) too -- multi-seed stats are saved either way in "
          "multi_seed_repeats_summary.csv, and will be folded in automatically once "
          "that file exists.")

print(f"\nSaved: {MULTISEED_DIR / 'multi_seed_repeats_raw.csv'}")
print(f"Saved: {MULTISEED_DIR / 'multi_seed_repeats_summary.csv'}")

In [ ]:
# =============================================================================
# MULTI-SEED COMPARISON FIGURE
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, metric in zip(axes, ["mcc", "macro_f1"]):
    if metric not in summary_stats.index:
        ax.axis("off")
        continue
    mean_val = summary_stats.loc[metric, "mean"]
    std_val = summary_stats.loc[metric, "std"]
    ax.bar([ACTIVE_DATASET], [mean_val], yerr=[std_val], capsize=8, color="#4C72B0")
    ax.set_title(f"Test {metric.upper()} — mean ± std ({N_REPEAT_SEEDS} seeds)")
    ax.set_ylim(0, 1)
    ax.text(0, mean_val + std_val + 0.02, f"{mean_val:.4f} ± {std_val:.4f}",
            ha="center", fontsize=10)

fig.tight_layout()
fig_path = FIGURES_DIR / "multi_seed_robustness.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

## Next steps
1. `python run_all.py --steps 2 3 --datasets nf_ton_iot_v2`: evaluation and baselines, then the C header, test vectors and sketch.
2. Flash `results/esp32_export/nf_ton_iot_v2/esp32_dt_test/esp32_dt_test.ino` and save the Serial output as `esp32/serial_logs/nf_ton_iot_v2_serial.txt`.
3. `python run_all.py --steps 5 6`: hardware analysis and figures.